## Project 1: Advanced EDA & Feature Engineering

In [2]:
import pandas as pd, numpy as np
from sklearn.impute import KNNImputer
from scipy.stats import zscore
df=pd.read_csv('Dataset for Data Analytics - Sheet1.csv')
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [3]:
nums=['Quantity','UnitPrice','ItemsInCart','TotalPrice']
for c in nums: df[c]=pd.to_numeric(df[c],errors='coerce')
df['Quantity']=df['Quantity'].fillna(df['Quantity'].mean())
df['UnitPrice']=df['UnitPrice'].fillna(df['UnitPrice'].median())
df[['ItemsInCart','TotalPrice']]=KNNImputer(n_neighbors=5).fit_transform(df[['ItemsInCart','TotalPrice']])
for c in ['CouponCode','ReferralSource','PaymentMethod','OrderStatus']:
    df[c]=df[c].fillna(df[c].mode()[0])

In [5]:
# Create Fraud column using business rules
df["Fraud"] = np.where(
    (
        (df["TotalPrice"] > 3000) &
        (df["Quantity"] >= 5)
    ) |
    (
        (df["ItemsInCart"] > 6) &
        (df["CouponCode"] == "No Coupon")
    ) |
    (
        (df["PaymentMethod"] == "Debit Card") &
        (df["TotalPrice"] > 2500)
    ),
    1,
    0
)

print(df["Fraud"].value_counts())

Fraud
0    1162
1      30
Name: count, dtype: int64


In [7]:
for c in nums:
 q1,q3=df[c].quantile([0.25,0.75]);iqr=q3-q1
 df=df[(df[c]>=q1-1.5*iqr)&(df[c]<=q3+1.5*iqr)]
z=np.abs(zscore(df[nums]));df=df[(z<3).all(axis=1)]
df['AverageItemPrice']=df['TotalPrice']/df['Quantity']
df['DiscountApplied']=np.where(df['CouponCode'].eq('No Coupon'),0,1)
df['CartValue']=df['ItemsInCart']*df['UnitPrice']
df.to_csv('clean_feature_engineered_orders.csv',index=False)
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice,AverageItemPrice,DiscountApplied,CartValue,Fraud
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7.0,SAVE10,Instagram,2853.10,570.62,1,3994.34,1
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3.0,SAVE10,Referral,302.70,151.35,1,454.05,0
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8.0,FREESHIP,Email,2753.40,550.68,1,4405.44,0
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5.0,SAVE10,Facebook,273.19,273.19,1,1365.95,0
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8.0,SAVE10,Email,2504.04,626.01,1,5008.08,0
